# Qdrant Medical Multi-Vector Query

Searches the `medical_facts_global` collection with the two active named dense vectors:

- `primary`: local fine-tuned SentenceTransformer in `storage/models/fine_tuned_medical_retriever`
- `sapbert`: `cambridgeltl/SapBERT-from-PubMedBERT-fulltext`

The embedding and normalization flow mirrors the ingestion scripts: `SentenceTransformer.encode(..., convert_to_numpy=True)` followed by L2 normalization.


In [ ]:
# Colab/local setup. Skip this cell if the packages are already installed.
!pip -q install qdrant-client sentence-transformers numpy tqdm


## Configuration


In [ ]:
import os
from pathlib import Path

QDRANT_URL = os.getenv('QDRANT_URL', '<YOUR_QDRANT_URL>').strip()
QDRANT_API_KEY = os.getenv('QDRANT_API_KEY', '<YOUR_QDRANT_API_KEY>').strip() or None
COLLECTION_NAME = os.getenv('COLLECTION_NAME', 'medical_facts_global_1')

# In Colab, set PROJECT_ROOT to your Drive project folder.
# Locally, this resolves correctly from either the repo root or retriever/.
PROJECT_ROOT = Path(os.getenv('PROJECT_ROOT', Path.cwd())).resolve()
if not (PROJECT_ROOT / 'storage').exists() and (PROJECT_ROOT.parent / 'storage').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SAPBERT_MODEL_NAME = os.getenv('SAPBERT_MODEL', 'cambridgeltl/SapBERT-from-PubMedBERT-fulltext')
FINE_TUNE_MODEL_PATH = Path(os.getenv('FINE_TUNE_MODEL_PATH', 'storage/models/fine_tuned_medical_retriever'))
if not FINE_TUNE_MODEL_PATH.is_absolute():
    FINE_TUNE_MODEL_PATH = (PROJECT_ROOT / FINE_TUNE_MODEL_PATH).resolve()

VECTOR_MODEL_CONFIG = {
    'primary': str(FINE_TUNE_MODEL_PATH),
    'sapbert': SAPBERT_MODEL_NAME,
}

QUERY = os.getenv(
    'QUERY',
    'Exercising regularly can help reduce the risk of heart disease and improve overall cardiovascular health.',
)

QUERY_POOL_SIZE = int(os.getenv('QUERY_POOL_SIZE', '100'))
RERANK_TOP_K = int(os.getenv('RERANK_TOP_K', '40'))
DISPLAY_TOP_K = int(os.getenv('DISPLAY_TOP_K', '10'))
USE_CROSS_ENCODER_RERANKER = os.getenv('USE_CROSS_ENCODER_RERANKER', 'true').lower() in {'1', 'true', 'yes', 'y'}

print(f'Project root: {PROJECT_ROOT}')
print(f'Collection: {COLLECTION_NAME}')
print(f'Query: {QUERY}')
print(f'Fine-tuned model path: {FINE_TUNE_MODEL_PATH}')


## Imports and Helpers


In [ ]:
from typing import Any, Dict, Iterable, List

import numpy as np
from qdrant_client import QdrantClient, models
from sentence_transformers import SentenceTransformer


def l2_normalize(vectors: np.ndarray) -> np.ndarray:
    if vectors.ndim == 1:
        norm = np.linalg.norm(vectors)
        if norm == 0:
            return vectors
        return vectors / norm

    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    norms[norms == 0] = 1e-10
    return vectors / norms


def min_max_normalize(values: Iterable[float]) -> List[float]:
    values = list(values)
    if not values:
        return []
    minimum = min(values)
    maximum = max(values)
    if maximum == minimum:
        return [0.0 for _ in values]
    return [(value - minimum) / (maximum - minimum) for value in values]


def payload_text(payload: Dict[str, Any]) -> str:
    parts = [
        payload.get('title') or '',
        payload.get('section_heading') or '',
        payload.get('text') or '',
    ]
    return '\n'.join(part for part in parts if part).strip()


def point_to_row(point: Any, retrieval_mode: str) -> Dict[str, Any]:
    payload = point.payload or {}
    return {
        'id': str(point.id),
        'retrieval_mode': retrieval_mode,
        'score': float(point.score),
        'source_site': payload.get('source_site'),
        'source_type': payload.get('source_type'),
        'title': payload.get('title'),
        'section_heading': payload.get('section_heading'),
        'text': payload.get('text'),
        'url': payload.get('url'),
        'payload': payload,
    }


def print_rows(rows: List[Dict[str, Any]], limit: int = DISPLAY_TOP_K) -> None:
    for rank, row in enumerate(rows[:limit], 1):
        print(f'\nResult {rank}')
        print(f"Mode: {row.get('retrieval_mode')}")
        if row.get('overall_score') is not None:
            print(f"Overall score: {row['overall_score']:.4f}")
        if row.get('cross_score') is not None:
            print(f"Cross score: {row['cross_score']:.4f}")
        print(f"Qdrant score: {row.get('score', 0.0):.4f}")
        print(f"Source: {row.get('source_site')} / {row.get('source_type')}")
        print(f"Title: {row.get('title')}")
        print(f"Section: {row.get('section_heading')}")
        print(f"URL: {row.get('url')}")
        print(f"Text: {row.get('text')}")


## Connect to Qdrant and Validate Vector Names


In [ ]:
qdrant_client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
collection_info = qdrant_client.get_collection(COLLECTION_NAME)

vectors_config = collection_info.config.params.vectors
if isinstance(vectors_config, dict):
    available_vectors = set(vectors_config.keys())
else:
    available_vectors = set()

required_vectors = set(VECTOR_MODEL_CONFIG.keys())
missing_vectors = sorted(required_vectors - available_vectors)
print(f'Available named vectors: {sorted(available_vectors)}')

if missing_vectors:
    raise RuntimeError(
        f'Missing required vector(s) in {COLLECTION_NAME}: {missing_vectors}. '
        'Rebuild the collection with the updated two-vector ingestion notebooks before querying.'
    )


## Load Models and Encode Query


In [ ]:
if not FINE_TUNE_MODEL_PATH.exists():
    raise FileNotFoundError(f'Fine-tuned model path not found: {FINE_TUNE_MODEL_PATH}')

embedding_models: Dict[str, SentenceTransformer] = {}
query_vectors: Dict[str, List[float]] = {}

for vector_name, model_name_or_path in VECTOR_MODEL_CONFIG.items():
    print(f'[Model] Loading {vector_name}: {model_name_or_path}')
    model = SentenceTransformer(model_name_or_path)
    embedding_models[vector_name] = model

    vector = model.encode(
        QUERY,
        convert_to_numpy=True,
        show_progress_bar=False,
    )
    vector = l2_normalize(vector)
    query_vectors[vector_name] = vector.astype(float).tolist()
    print(f'[Model] {vector_name} dimension: {len(query_vectors[vector_name])}')


## Individual Vector Searches


In [ ]:
def search_single_vector(vector_name: str, limit: int = QUERY_POOL_SIZE):
    return qdrant_client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vectors[vector_name],
        using=vector_name,
        limit=limit,
        with_payload=True,
        timeout=90,
    )

individual_results = {}
individual_rows = {}

for vector_name in ['primary', 'sapbert']:
    print(f'\n[Qdrant] Searching vector: {vector_name}')
    result = search_single_vector(vector_name, limit=QUERY_POOL_SIZE)
    individual_results[vector_name] = result
    individual_rows[vector_name] = [point_to_row(point, vector_name) for point in result.points]
    print_rows(individual_rows[vector_name], limit=5)


## RRF Fusion Search Across Primary and SapBERT


In [ ]:
fused_results = qdrant_client.query_points(
    collection_name=COLLECTION_NAME,
    prefetch=[
        models.Prefetch(query=query_vectors['primary'], using='primary', limit=QUERY_POOL_SIZE),
        models.Prefetch(query=query_vectors['sapbert'], using='sapbert', limit=QUERY_POOL_SIZE),
    ],
    query=models.FusionQuery(fusion=models.Fusion.RRF),
    limit=QUERY_POOL_SIZE,
    with_payload=True,
    timeout=90,
)

fused_rows = [point_to_row(point, 'rrf_primary_fine_tuned_plus_sapbert') for point in fused_results.points]
print_rows(fused_rows, limit=DISPLAY_TOP_K)


## Optional Cross-Encoder Reranking


In [ ]:
if USE_CROSS_ENCODER_RERANKER:
    from sentence_transformers.cross_encoder import CrossEncoder

    cross_encoder = CrossEncoder('ncbi/MedCPT-Cross-Encoder', max_length=512)
    candidate_rows = fused_rows[:RERANK_TOP_K]
    candidate_texts = [payload_text(row['payload']) for row in candidate_rows]

    usable_pairs = [
        (row, text)
        for row, text in zip(candidate_rows, candidate_texts)
        if text
    ]
    if not usable_pairs:
        raise ValueError('No candidate text found for cross-encoder reranking.')

    rerank_rows = [row.copy() for row, _ in usable_pairs]
    rerank_texts = [text for _, text in usable_pairs]

    cross_scores = cross_encoder.predict(
        [[QUERY, text] for text in rerank_texts],
        batch_size=16,
        show_progress_bar=True,
    )
    cross_scores = [float(score) for score in cross_scores]
    qdrant_scores = [float(row['score']) for row in rerank_rows]

    qdrant_scores_norm = min_max_normalize(qdrant_scores)
    cross_scores_norm = min_max_normalize(cross_scores)

    qdrant_weight = 0.35
    cross_encoder_weight = 0.65

    reranked_rows = []
    for row, qdrant_norm, cross_norm, cross_score in zip(
        rerank_rows,
        qdrant_scores_norm,
        cross_scores_norm,
        cross_scores,
    ):
        row['qdrant_score_norm'] = float(qdrant_norm)
        row['cross_score_norm'] = float(cross_norm)
        row['cross_score'] = float(cross_score)
        row['overall_score'] = (
            qdrant_weight * row['qdrant_score_norm']
            + cross_encoder_weight * row['cross_score_norm']
        )
        row['retrieval_mode'] = 'rrf_all_vectors_plus_cross_encoder'
        reranked_rows.append(row)

    reranked_rows.sort(key=lambda item: item['overall_score'], reverse=True)
    print_rows(reranked_rows, limit=DISPLAY_TOP_K)
else:
    reranked_rows = []
    print('Cross-encoder reranking disabled.')


## Clean Result Objects


In [ ]:
source_rows = reranked_rows if reranked_rows else fused_rows

clean_results = []
for row in source_rows:
    clean_results.append({
        'id': row.get('id'),
        'retrieval_mode': row.get('retrieval_mode'),
        'score': row.get('overall_score', row.get('score')),
        'qdrant_score': row.get('score'),
        'cross_score': row.get('cross_score'),
        'source_site': row.get('source_site'),
        'source_type': row.get('source_type'),
        'title': row.get('title'),
        'section_heading': row.get('section_heading'),
        'text': row.get('text'),
        'url': row.get('url'),
    })

clean_results[:DISPLAY_TOP_K]
